In [14]:
import pandas as pd
import numpy as np
import math
import pickle

In [15]:
df = pd.read_pickle("../data/clean.pkl")
display(df.head())

,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,power,overstrain,temperature_difference
0,1,298.1,308.6,1551,42.8,0,0,6952.0,0.0,10.0
1,0,298.2,308.7,1408,46.3,3,0,6827.0,139.0,10.0
2,0,298.1,308.5,1498,49.4,5,0,7749.0,247.0,10.0
3,0,298.2,308.6,1433,39.5,7,0,5928.0,276.0,10.0
4,0,298.2,308.7,1408,40.0,9,0,5898.0,360.0,10.0


In [17]:
#split into two df on machine failure 0/1

training_ratio = 0.65
testing_ratio = 1 - training_ratio

failures = df[df["machine_failure"] == 1]
normal = df[df["machine_failure"] == 0]

display(normal.head())
display(failures.head())

print(len(failures))

training_normal = normal.sample(frac=0.65, random_state=42)
testing_normal = normal.drop(training_normal.index)

training_failures = failures.sample(frac=0.65, random_state=42)
testing_failures = failures.drop(training_failures.index)

training = pd.concat([training_normal,training_failures] , ignore_index=True)
testing = pd.concat([testing_normal, testing_failures] , ignore_index=True)

print(f"Number of failures {training["machine_failure"].sum()} , percent of failures {training["machine_failure"].sum() / len(training)} in training")
print(f"Number of failures {testing["machine_failure"].sum()} , percent of failures {testing["machine_failure"].sum()/len(testing)} in testing")


,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,power,overstrain,temperature_difference
0,1,298.1,308.6,1551,42.8,0,0,6952.0,0.0,10.0
1,0,298.2,308.7,1408,46.3,3,0,6827.0,139.0,10.0
2,0,298.1,308.5,1498,49.4,5,0,7749.0,247.0,10.0
3,0,298.2,308.6,1433,39.5,7,0,5928.0,276.0,10.0
4,0,298.2,308.7,1408,40.0,9,0,5898.0,360.0,10.0


,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,power,overstrain,temperature_difference
50,0,298.9,309.1,2861,4.6,143,1,1378.0,658.0,10.0
69,0,298.9,309.0,1410,65.7,191,1,9701.0,12549.0,10.0
77,0,298.8,308.9,1455,41.3,208,1,6293.0,8590.0,10.0
160,0,298.4,308.2,1282,60.7,216,1,8149.0,13111.0,10.0
161,0,298.3,308.1,1412,52.3,218,1,7733.0,11401.0,10.0


339
Number of failures 220 , percent of failures 0.033846153846153845 in training
Number of failures 119 , percent of failures 0.034 in testing


Evaluation strategy:

Primary metric: F2 score, recall weighted more heavily than precison as missed failures are more costly than false alarms
Secondary metric: PR-AUC, to assess performance across all thresholds not just at 0.5
Supporting data:
- Confusion matrix
- Cross-validation with stratifiedKfold with k = 5. Splitting training data into 5 chunks training using 4 of them and testing on the 5th repeat until all of the 5 chunks have been the mock test data 
Thus the above is just an exmaple of how data could be split. But I will divide it into 5 chunks and construct data frames for training and testing using them. Will move from 65% training data as it is awkward to do with the chunks.
